# Data Generation — Safety-Aligned Distillation

This notebook runs the teacher model (Llama-3.1-8B-Instruct) on benign and harmful prompts
to generate the distillation dataset.

**Before running:**
1. Enable GPU: Notebook settings → Accelerator → **T4 x2** or **P100**
2. Enable Internet: Notebook settings → Internet → **On**
3. Add your HuggingFace token: Add-ons → Secrets → Add **HF_TOKEN**

## Step 0: Install Dependencies

In [ ]:
!pip install -q datasets transformers accelerate huggingface_hub tqdm

## Step 1: Setup

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# Load HF token from Kaggle secrets
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

from huggingface_hub import login
login(token=hf_token)
print("HuggingFace login successful.")

In [ ]:
import json
import re
from pathlib import Path
import torch

print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## Step 2: Configuration

In [ ]:
# ── Config ──────────────────────────────────────────────
TEACHER_MODEL  = "meta-llama/Llama-3.1-8B-Instruct"
N_BENIGN       = 5000
N_HARMFUL      = 500
MAX_NEW_TOKENS = 512
BATCH_SIZE     = 8     # increase if you have more VRAM
OUTPUT_PATH    = Path("/kaggle/working/raw_teacher_outputs.jsonl")
PROCESSED_DIR  = Path("/kaggle/working/processed")
# ────────────────────────────────────────────────────────

SYSTEM_PROMPT = "You are a helpful, harmless, and honest assistant."

REFUSAL_PREFIXES = [
    "I'm sorry", "I can't", "I cannot", "I'm not able", "I won't",
    "I will not", "As an AI", "I'm unable", "I apologize",
    "That's not something I", "I don't think I should",
    "This request involves", "I must decline",
]

print(f"Teacher model  : {TEACHER_MODEL}")
print(f"Benign prompts : {N_BENIGN}")
print(f"Harmful prompts: {N_HARMFUL}")
print(f"Batch size     : {BATCH_SIZE}")

## Step 3: Helper Functions

In [ ]:
def is_refusal(text: str) -> bool:
    text_lower = text.lower().strip()
    for prefix in REFUSAL_PREFIXES:
        if text_lower.startswith(prefix.lower()):
            return True
    patterns = [
        r"i (am|'m) (not able|unable) to",
        r"i (can't|cannot|won't|will not) (help|assist|provide|generate|create)",
        r"(this|that) (request|prompt|query) (is|seems|appears) (harmful|dangerous|unethical|illegal)",
    ]
    for pattern in patterns:
        if re.search(pattern, text_lower):
            return True
    return False


def save_jsonl(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} records → {path}")


print("Helper functions ready.")

## Step 4: Load Prompts

In [ ]:
import urllib.request
import csv
import io
from datasets import load_dataset

# Benign prompts from Alpaca
print(f"Loading {N_BENIGN} benign prompts from tatsu-lab/alpaca ...")
alpaca = load_dataset("tatsu-lab/alpaca", split="train")
benign_prompts = []
for row in alpaca:
    instruction = row["instruction"].strip()
    inp = row["input"].strip()
    user_turn = f"{instruction}\n\n{inp}" if inp else instruction
    if len(user_turn) > 20:
        benign_prompts.append(user_turn)
    if len(benign_prompts) >= N_BENIGN:
        break
print(f"  Loaded {len(benign_prompts)} benign prompts.")

# Harmful prompts from AdvBench (public CSV — no gating)
ADVBENCH_URL = (
    "https://raw.githubusercontent.com/llm-attacks/llm-attacks/"
    "main/data/advbench/harmful_behaviors.csv"
)
print(f"\nLoading {N_HARMFUL} harmful prompts from AdvBench CSV ...")
with urllib.request.urlopen(ADVBENCH_URL) as response:
    content = response.read().decode("utf-8")

reader = csv.DictReader(io.StringIO(content))
harmful_prompts = []
for row in reader:
    prompt = row["goal"].strip()
    if prompt:
        harmful_prompts.append(prompt)
    if len(harmful_prompts) >= N_HARMFUL:
        break
print(f"  Loaded {len(harmful_prompts)} harmful prompts.")

## Step 5: Load Teacher Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

print(f"Loading teacher model: {TEACHER_MODEL}")
print("This may take 5-10 minutes...")

tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",   # spreads across all available GPUs automatically
)
model.eval()
print("Teacher model loaded.")
print(f"Model device map: {model.hf_device_map if hasattr(model, 'hf_device_map') else 'single GPU'}")


def run_inference(prompts: list[str], category: str) -> list[dict]:
    """Run batched inference and return records."""
    records = []
    for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc=f"{category}"):
        batch_prompts = prompts[i: i + BATCH_SIZE]

        # Apply Llama-3 chat template via tokenizer
        chats = [[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": p},
        ] for p in batch_prompts]

        inputs_text = [
            tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
            for chat in chats
        ]

        inputs = tokenizer(
            inputs_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,        # greedy decoding
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Decode only the newly generated tokens
        input_len = inputs["input_ids"].shape[1]
        for prompt, output_ids in zip(batch_prompts, outputs):
            response = tokenizer.decode(
                output_ids[input_len:], skip_special_tokens=True
            ).strip()
            records.append({
                "prompt": prompt,
                "response": response,
                "category": category,
                "is_refusal": is_refusal(response) if category == "harmful" else False,
            })

    return records

## Step 6: Run Inference on Benign Prompts

In [ ]:
print("Running inference on benign prompts (~3–5 hours on T4)...")
benign_records = run_inference(benign_prompts, "benign")
print(f"Done. Generated {len(benign_records)} benign records.")

## Step 7: Run Inference on Harmful Prompts

In [ ]:
print("Running inference on harmful prompts (~20 min on T4)...")
harmful_records = run_inference(harmful_prompts, "harmful")

refusals = sum(1 for r in harmful_records if r["is_refusal"])
print(f"Done. Generated {len(harmful_records)} harmful records.")
print(f"Refusals: {refusals}/{len(harmful_records)} ({100*refusals/len(harmful_records):.1f}%)")

## Step 8: Save Raw Outputs

In [ ]:
all_records = benign_records + harmful_records
save_jsonl(all_records, OUTPUT_PATH)

print(f"\n--- Dataset Statistics ---")
print(f"  Total   : {len(all_records)}")
print(f"  Benign  : {len(benign_records)}")
print(f"  Harmful : {len(harmful_records)}")
print(f"  Refusals: {refusals}")

## Step 9: Prepare Train/Val Splits

In [ ]:
import random

SYSTEM_PROMPT = "You are a helpful, harmless, and honest assistant."

def format_sft_example(prompt, response):
    text = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n{response}<|eot_id|>"
    )
    return {"text": text, "prompt": prompt, "response": response}

# Filter: drop harmful compliances
filtered = [r for r in all_records if not (r["category"] == "harmful" and not r["is_refusal"])]
dropped = len(all_records) - len(filtered)
print(f"Dropped {dropped} harmful compliance records.")
print(f"Kept {len(filtered)} records.")

# Format into SFT template
formatted = []
for r in filtered:
    ex = format_sft_example(r["prompt"], r["response"])
    ex["category"] = r["category"]
    ex["is_refusal"] = r["is_refusal"]
    formatted.append(ex)

# Shuffle and split
rng = random.Random(42)
rng.shuffle(formatted)
split_idx = int(len(formatted) * 0.95)
train_records = formatted[:split_idx]
val_records = formatted[split_idx:]

print(f"Train: {len(train_records)}  |  Val: {len(val_records)}")

# Save
save_jsonl(train_records, PROCESSED_DIR / "train.jsonl")
save_jsonl(val_records,   PROCESSED_DIR / "val.jsonl")
print("\nDataset preparation complete.")

## Step 10: Verify & Preview

In [ ]:
# Preview a benign example
benign_ex = next(r for r in train_records if r["category"] == "benign")
print("=== Benign Example ===")
print(f"Prompt  : {benign_ex['prompt'][:100]}...")
print(f"Response: {benign_ex['response'][:150]}...")

print()

# Preview a refusal example
refusal_ex = next(r for r in train_records if r["is_refusal"])
print("=== Refusal Example ===")
print(f"Prompt  : {refusal_ex['prompt'][:100]}...")
print(f"Response: {refusal_ex['response'][:150]}...")

print()
print("All files saved to /kaggle/working/")
print("Download them from the Output tab on the right.")